# 06 — Console Views

The 0.3.1 release added a set of **lightweight dataclasses** and
single-query methods tuned for web-console consumption. These
dataclasses serialize cleanly to JSON, avoid per-holon fan-out
queries, and shape payloads for common graph-visualization libraries.

This notebook covers:

- `list_holons_summary()` — cheap single-query holon listing
- `get_holon_detail(iri)` — full descriptor
- `holon_neighborhood(iri, depth=N)` — BFS subgraph for sigma.js
- `to_graphology()` — the payload shape for graphology/sigma.js
- `list_portals()` / `get_portal(iri)` — flat portal browsing
- `portal_traversal_history(iri)` — scoped PROV-O feed


## Setup — a small holarchy with some activity


In [ ]:
from holonic import HolonicDataset
from dataclasses import asdict
import json

ds = HolonicDataset()

# Build a small holarchy
for name in ("alpha", "beta", "gamma"):
    ds.add_holon(f"urn:holon:{name}", name.title())
    ds.add_interior(f"urn:holon:{name}", f'''
        @prefix ex: <urn:ex:> .
        <urn:entity:{name}:1> a ex:Widget ; ex:label "{name}-1" .
        <urn:entity:{name}:2> a ex:Gadget ; ex:label "{name}-2" .
    ''')

# A couple of portals
ds.add_portal("urn:portal:a-to-b", "urn:holon:alpha", "urn:holon:beta",
              "CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
              label="Alpha → Beta")
ds.add_portal("urn:portal:b-to-g", "urn:holon:beta", "urn:holon:gamma",
              "CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
              label="Beta → Gamma")


## Lightweight listing

`list_holons_summary()` returns `HolonSummary` records via a single
SPARQL query. No per-holon fan-out. Contrast with `list_holons()`
which returns richer `HolonInfo` at fan-out cost.


In [ ]:
summaries = ds.list_holons_summary()
for s in summaries:
    print(f"{s.iri:30s}  {s.label:12s}  class={s.kind}")


## Full holon detail

`get_holon_detail(iri)` returns a `HolonDetail` with layer graph IRIs
and an interior triple count. Since 0.3.3 it also surfaces
per-layer `GraphMetadata` when available (see `07_graph_metadata`).


In [ ]:
detail = ds.get_holon_detail("urn:holon:alpha")
print(json.dumps(asdict(detail), indent=2, default=str))


## Class inventory on one holon

`holon_interior_classes(iri)` returns `(class_iri, count)` pairs
computed by SPARQL. Useful for "what kinds of things are in here?"
panels.


In [ ]:
for entry in ds.holon_interior_classes("urn:holon:alpha"):
    print(f"{entry.class_iri:40s}  {entry.count}")


## Neighborhood subgraph (graphology-compatible)

`holon_neighborhood(iri, depth=N)` returns a `NeighborhoodGraph`
with bounded BFS from the focus holon. `depth` is clamped to keep
runaway traversals in check.


In [ ]:
neighborhood = ds.holon_neighborhood("urn:holon:beta", depth=2)
print(f"nodes: {len(neighborhood.nodes)}  edges: {len(neighborhood.edges)}")
print()
print("Raw:")
for n in neighborhood.nodes:
    print(f"  {n.key}  label={n.label!r}")
for e in neighborhood.edges:
    print(f"  {e.source}  →  {e.target}  ({e.label})")


## Graphology-ready JSON

`NeighborhoodGraph.to_graphology()` returns the payload shape that
sigma.js + graphology consume directly:


In [ ]:
payload = neighborhood.to_graphology()
print(json.dumps(payload, indent=2)[:1000])
print("...")


## Portal browsing (flat, not keyed by source/target)

`list_portals()` and `get_portal(iri)` give a flat portal catalog,
separate from the `find_portals_from`/`find_portals_to` family.


In [ ]:
portals = ds.list_portals()
for p in portals:
    print(f"{p.iri:30s}  {p.label!r}")


## Portal traversal history

`portal_traversal_history(iri, limit=N)` returns recent
`prov:Activity` records scoped to one portal. Limit is clamped to
10,000 to bound runaway queries.


In [ ]:
# First, generate some traversals
ds.traverse("urn:holon:alpha", "urn:holon:beta", agent_iri="urn:agent:op-1")
ds.traverse("urn:holon:alpha", "urn:holon:beta", agent_iri="urn:agent:op-2")

history = ds.portal_traversal_history("urn:portal:a-to-b", limit=10)
for record in history:
    print(f"{record.timestamp}  agent={record.agent_iri}")


## Where to go next

- `07_graph_metadata` — per-graph triple counts and class inventory
  that feed into `get_holon_detail.layer_metadata`
- `08_scope_resolution` — find holons matching predicates across
  the holarchy without per-neighborhood queries
